# End-to-End Multilayer Network Analysis with Py3Plex

This notebook demonstrates a complete end-to-end workflow for analyzing multilayer networks using py3plex. We'll cover:

1. **Network Creation** - Building a multilayer network from scratch
2. **Data Loading** - Loading networks from files
3. **Network Exploration** - Computing basic statistics and structure
4. **Community Detection** - Finding communities in multilayer networks
5. **Centrality Analysis** - Computing various centrality measures
6. **Visualization** - Creating meaningful visualizations
7. **Advanced Analysis** - Multilayer-specific metrics

This example provides a complete workflow that can be adapted for your own research questions.

## 1. Setup and Imports

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Py3plex imports
from py3plex.core import multinet
from py3plex.core import random_generators
from py3plex.algorithms.community_detection import community_wrapper as cw
from py3plex.algorithms.multilayer_algorithms.centrality import compute_all_centralities
from py3plex.visualization.multilayer import hairball_plot, draw_multilayer_default
from py3plex.visualization.colors import colors_default

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All imports successful")

## 2. Network Creation

Let's start by creating a multilayer network from scratch. We'll build a simple social-professional network with three layers.

In [ ]:
# Create a new multilayer network
network = multinet.multi_layer_network(directed=False)

# Layer 1: Social network (friendship connections)
social_edges = [
    ['Alice', 'social', 'Bob', 'social', 1],
    ['Bob', 'social', 'Charlie', 'social', 1],
    ['Charlie', 'social', 'Alice', 'social', 1],
    ['Alice', 'social', 'David', 'social', 1],
    ['David', 'social', 'Eve', 'social', 1],
    ['Eve', 'social', 'Frank', 'social', 1],
    ['Frank', 'social', 'Bob', 'social', 1]
]

# Layer 2: Professional network (work collaboration)
professional_edges = [
    ['Alice', 'professional', 'Bob', 'professional', 2],
    ['Alice', 'professional', 'Charlie', 'professional', 2],
    ['Bob', 'professional', 'David', 'professional', 2],
    ['Charlie', 'professional', 'David', 'professional', 2],
    ['Eve', 'professional', 'Frank', 'professional', 2]
]

# Layer 3: Communication network (email/messaging)
communication_edges = [
    ['Alice', 'communication', 'Bob', 'communication', 1],
    ['Alice', 'communication', 'Charlie', 'communication', 1],
    ['Alice', 'communication', 'David', 'communication', 1],
    ['Bob', 'communication', 'Eve', 'communication', 1],
    ['David', 'communication', 'Frank', 'communication', 1]
]

# Add all edges to the network
all_edges = social_edges + professional_edges + communication_edges
network.add_edges(all_edges, input_type='list')

print("✓ Multilayer network created successfully")
print(f"  - Total node-layer tuples: {len(list(network.get_nodes()))}")
print(f"  - Unique nodes: {len(set([n[0] for n in network.get_nodes()]))}")
print(f"  - Total layers: {len(list(network.get_layers()))}")
print(f"  - Layers: {list(network.get_layers())}")

## 3. Basic Network Statistics

Let's explore the structure of our network.

In [ ]:
# Display comprehensive network statistics
print("Network Statistics:")
print("=" * 50)
network.basic_stats()

# Get more detailed information
nodes = list(network.get_nodes())
layers = list(network.get_layers())

# Get unique node IDs (across all layers)
unique_node_ids = set([node[0] for node in nodes])

print("\nDetailed Information:")
print("=" * 50)
print(f"Number of node-layer tuples: {len(nodes)}")
print(f"Unique node IDs across layers: {sorted(unique_node_ids)}")
print(f"Number of unique node IDs: {len(unique_node_ids)}")
print(f"Number of layers: {len(layers)}")
print(f"Layer names: {layers}")

# Count edges per layer
print("\nEdges per layer:")
for layer in layers:
    layer_edges = [(u, v) for u, v, data in network.core_network.edges(data=True) 
                   if data.get('type') == layer]
    print(f"  {layer}: {len(layer_edges)} edges")

## 4. Network Visualization

Visualizing the network helps us understand its structure at a glance.

In [ ]:
# Basic hairball visualization
plt.figure(figsize=(10, 8))
hairball_plot(
    network.core_network,
    layout_algorithm="force",
    layout_parameters={"iterations": 100},
    scale_by_size=True,
    legend=True
)
plt.title("Multilayer Network - Hairball Visualization")
plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## 5. Community Detection

Let's identify communities within our multilayer network using the Louvain algorithm.

In [ ]:
# Detect communities using Louvain algorithm
partition = cw.louvain_communities(network)

# Analyze community structure
community_counts = Counter(partition.values())
num_communities = len(community_counts)

print(f"Community Detection Results:")
print("=" * 50)
print(f"Number of communities found: {num_communities}")
print(f"\nCommunity sizes:")
for community_id, size in sorted(community_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  Community {community_id}: {size} nodes")

# Show which nodes belong to which community
print(f"\nNode-to-Community mapping (first few nodes):")
for node in sorted(nodes)[:10]:  # Show first 10 node-layer tuples
    if node in partition:
        print(f"  {node}: Community {partition[node]}")

## 6. Visualize Communities

Now let's visualize the network with communities colored.

In [ ]:
# Create color mapping for communities
unique_communities = list(set(partition.values()))
color_map = dict(zip(unique_communities, colors_default[:len(unique_communities)]))

# Assign colors to nodes based on their community
node_colors = [color_map[partition.get(node, 0)] for node in network.get_nodes()]

# Visualize with community colors
plt.figure(figsize=(12, 8))
hairball_plot(
    network.core_network,
    color_list=node_colors,
    layout_algorithm="force",
    layout_parameters={"iterations": 100},
    scale_by_size=True,
    legend=True
)
plt.title("Multilayer Network with Community Detection")
plt.tight_layout()
plt.show()

print("✓ Community visualization complete")

## 7. Centrality Analysis

Let's compute various centrality measures to identify important nodes in the network.

In [ ]:
# Compute all available centrality measures
print("Computing centrality measures...")
centralities = compute_all_centralities(network)

print("\nCentrality Analysis Results:")
print("=" * 50)

# Display results for each centrality measure
for centrality_name, centrality_values in centralities.items():
    print(f"\n{centrality_name.upper()}:")
    # Sort nodes by centrality value (descending)
    sorted_nodes = sorted(centrality_values.items(), key=lambda x: x[1], reverse=True)
    for node, value in sorted_nodes:
        print(f"  {str(node):15} {value:.4f}")

## 8. Generate a Larger Random Network

For more advanced analysis, let's generate a larger random multilayer network.

In [ ]:
# Generate a random Erdős-Rényi multilayer network
print("Generating random multilayer network...")
random_network = random_generators.random_multilayer_ER(
    n=50,              # Number of nodes
    l=3,               # Number of layers
    p=0.1,             # Probability of edge creation
    directed=False
)

print("\n✓ Random network generated")
print("\nRandom Network Statistics:")
print("=" * 50)
random_network.basic_stats()

## 9. Analyze the Random Network

In [ ]:
# Detect communities in the random network
random_partition = cw.louvain_communities(random_network)
random_community_counts = Counter(random_partition.values())

print("Random Network Community Structure:")
print("=" * 50)
print(f"Number of communities: {len(random_community_counts)}")
print(f"\nCommunity size distribution:")
for comm_id, size in sorted(random_community_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  Community {comm_id}: {size} nodes")

## 10. Visualize the Random Network

In [ ]:
# Create color mapping for random network communities
top_n = min(10, len(random_community_counts))
top_communities = list(random_community_counts.keys())[:top_n]
color_mappings = dict(zip(top_communities, colors_default[:top_n]))

random_colors = [
    color_mappings.get(random_partition.get(node, -1), 'black')
    for node in random_network.get_nodes()
]

# Visualize the random network
plt.figure(figsize=(14, 10))
hairball_plot(
    random_network.core_network,
    color_list=random_colors,
    layout_algorithm="force",
    layout_parameters={"iterations": 100},
    scale_by_size=True,
    legend=False
)
plt.title(f"Random Multilayer Network (50 nodes, 3 layers)")
plt.tight_layout()
plt.show()

print("✓ Random network visualization complete")

## 11. Summary and Key Insights

Let's summarize what we've learned from this end-to-end analysis.

In [ ]:
print("\n" + "="*60)
print("END-TO-END ANALYSIS SUMMARY")
print("="*60)

print("\n1. MANUAL NETWORK:")
print(f"   - Node-layer tuples: {len(list(network.get_nodes()))}")
print(f"   - Unique nodes: {len(set([n[0] for n in network.get_nodes()]))}")
print(f"   - Layers: {len(list(network.get_layers()))}")
print(f"   - Communities detected: {len(community_counts)}")

print("\n2. RANDOM NETWORK:")
print(f"   - Node-layer tuples: {len(list(random_network.get_nodes()))}")
print(f"   - Unique nodes: {len(set([n[0] for n in random_network.get_nodes()]))}")
print(f"   - Layers: {len(list(random_network.get_layers()))}")
print(f"   - Communities detected: {len(random_community_counts)}")

print("\n3. KEY CAPABILITIES DEMONSTRATED:")
print("   ✓ Manual network construction")
print("   ✓ Random network generation")
print("   ✓ Network statistics and exploration")
print("   ✓ Community detection (Louvain)")
print("   ✓ Centrality analysis (multiple metrics)")
print("   ✓ Network visualization")
print("   ✓ Multilayer-specific analysis")

print("\n4. NEXT STEPS:")
print("   - Load your own network data")
print("   - Experiment with different parameters")
print("   - Try other community detection algorithms")
print("   - Explore additional visualization options")
print("   - Apply to real-world research questions")

print("\n" + "="*60)
print("✓ End-to-end analysis complete!")
print("="*60)

## Conclusion

This notebook has demonstrated a complete end-to-end workflow for analyzing multilayer networks with py3plex. The workflow included:

1. **Network Creation**: Building networks from scratch with multiple layers
2. **Statistics**: Computing and understanding network properties
3. **Community Detection**: Identifying groups of related nodes
4. **Centrality**: Finding important nodes across layers
5. **Visualization**: Creating meaningful visual representations
6. **Random Networks**: Generating and analyzing synthetic networks

### Further Reading

- **Documentation**: [https://skblaz.github.io/py3plex/](https://skblaz.github.io/py3plex/)
- **More Examples**: Check the `examples/` directory in the repository
- **Research Paper**: Škrlj et al. (2019), "Py3plex toolkit for visualization and analysis of multilayer networks", Applied Network Science

### Adapting This Workflow

To adapt this workflow for your own research:

1. Replace the manual network creation with loading your own data
2. Adjust visualization parameters to suit your network size
3. Choose appropriate centrality measures for your domain
4. Experiment with different community detection algorithms
5. Add domain-specific analysis steps as needed

Happy analyzing! 🎉